# 📌 Submission Guidelines

### ✅ File Naming Rule  
Submit your notebook with the following format:

`StudentID_YourFullName_LABXX.ipynb`

or

`StudentID_YourFullName_LABXX.html`

**Examples (Correct):**
- `21113456_NguyenVanA_LAB01.ipynb`
- `19122233_TranThiB_LAB02.ipynb`

**Examples (Wrong → 0 points):**
- `Lab01.ipynb`
- `YourName.ipynb`
- `20123456.ipynb`
- `Lab01.pdf`

---

### 📌 Grading Policy

- ❌ Wrong filename format, missing submission, or plagiarism (code identical to others) → 0 points

- ⚠️ Submitting a file without results, incomplete work, or only the assignment description → Maximum 4 points

- ✅ Correct filename + Completed results → Graded normally based on assignment quality (accuracy, clarity, and originality)  
### Note:
- AI assistance is allowed, but you must write the code yourself. All submissions will be checked for originality.

---



## 🎓 Learning Objectives

After this lab, students will be able to:

- Understand the limitations of RNN/LSTM  
- Build an intuitive understanding of Self-Attention  
- Understand the overall architecture of the Transformer  
- Use a pretrained Transformer (e.g., BERT) for simple NLP tasks  


## 🧠 Big Picture Workflow (MOST IMPORTANT)

### 🔄 Transformer Pipeline

Text → Word Embedding → Positional Encoding → Self-Attention
→ Multi-head Attention → Transformer Encoder → NLP Task


### 💡 Intuitive Explanation

- **Word Embedding**: Convert words into vectors  
- **Positional Encoding**: Add positional information (since Transformers do not process sequences sequentially)  
- **Self-Attention**: Each word attends to all other words in the sentence  
- **Multi-head Attention**: Capture different types of relationships from multiple perspectives  
- **Transformer Encoder**: Combine all components into a unified representation  
- **Output**: Used for downstream tasks (e.g., classification, QA, etc.)


### 🎯 Key Idea

> - Instead of processing words sequentially like RNNs,  
> - Transformers process the entire sentence **in parallel**.

paper: http://arxiv.org/pdf/1706.03762

##SHORT THEORY (INTUITIVE)
Limitations of RNN/LSTM
- Sequential → slow
- Hard to capture long-range dependencies
- Limited parallelization
Self-Attention Idea

Each word asks:

“Which other words should I pay attention to?”

### Query, Key, Value (Intuition)
- Query (Q) → what this word is looking for
- Key (K) → what each word offers
- Value (V) → actual information

👉 Attention = compare Q with K → use corresponding V

Attention Score
- Measures similarity between words
- Use softmax to normalize

### Multi-head Attention
Instead of one attention:
- Head 1 → grammar
- Head 2 → meaning
- Head 3 → relationships

👉 Multiple “views” of the sentence

### Positional Encoding
Transformers don’t read in order → we must add position info:
- which word comes first?
- which comes later?

## Code Sample

In [ ]:
!pip install transformers
!pip install torch
!pip install datasets

In [ ]:
from datasets import load_dataset

dataset = load_dataset("imdb")

train_data = dataset["train"]
test_data = dataset["test"]

print(train_data[0])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


{'text': 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far be

In [ ]:
import torch
import torch.nn.functional as F

X = torch.rand(3, 4)
print(X)

Q = X @ torch.rand(4, 4)
print(Q)

K = X @ torch.rand(4, 4)
print(K)

V = X @ torch.rand(4, 4)
print(V)

scores = Q @ K.T
weights = F.softmax(scores, dim=1)

output = weights @ V

print(weights)

tensor([[0.0571, 0.3639, 0.1220, 0.1269],
        [0.1052, 0.8457, 0.3876, 0.3275],
        [0.0877, 0.4750, 0.4408, 0.0130]])
tensor([[0.4315, 0.2015, 0.1982, 0.2995],
        [1.0850, 0.5253, 0.4886, 0.7710],
        [0.5707, 0.3941, 0.1405, 0.5714]])
tensor([[0.4988, 0.1195, 0.3229, 0.2881],
        [1.2122, 0.2993, 0.7673, 0.7006],
        [0.7981, 0.1028, 0.4718, 0.3851]])
tensor([[0.3908, 0.4915, 0.2933, 0.2522],
        [0.9958, 1.2353, 0.7069, 0.6335],
        [0.6676, 0.7384, 0.3118, 0.3754]])
tensor([[0.2534, 0.4418, 0.3048],
        [0.1502, 0.6112, 0.2387],
        [0.2250, 0.4889, 0.2862]])


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
def tokenize(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_data = train_data.map(tokenize, batched=True)
test_data = test_data.map(tokenize, batched=True)

train_data.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_data.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

Map:   0%|          | 0/25000 [00:00<?, ? examples/s]

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(test_data, batch_size=32)

In [ ]:
from torch.optim import AdamW

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

optimizer = AdamW(model.parameters(), lr=5e-5)

In [ ]:
from tqdm.auto import tqdm

epochs = 20

for epoch in range(epochs):
    model.train()
    total_loss = 0

    # Wrap train_loader with tqdm for a progress bar
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        batch = {k: v.to(device) for k, v in batch.items()}
        batch["labels"] = batch.pop("label")

        outputs = model(**batch)
        loss = outputs.loss

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1:   0%|          | 0/782 [00:00<?, ?it/s]

Epoch 1, Loss: 265.8240


Epoch 2:   0%|          | 0/782 [00:00<?, ?it/s]

Epoch 2, Loss: 149.5927


Epoch 3:   0%|          | 0/782 [00:00<?, ?it/s]

...

Epoch 18, Loss: 8.5229


Epoch 19:   0%|          | 0/782 [00:00<?, ?it/s]

Epoch 19, Loss: 9.0619


Epoch 20:   0%|          | 0/782 [00:00<?, ?it/s]

Epoch 20, Loss: 10.6316


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)
        preds = torch.argmax(outputs.logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(batch["label"].cpu().numpy())

# Metrics
acc = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds)
recall = recall_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)

print(f"Accuracy: {acc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-score: {f1:.4f}")

Accuracy: 0.8363
Precision: 0.7876
Recall: 0.9210
F1-score: 0.8491


In [ ]:
text = "This movie is amazing!"

inputs = tokenizer(text, return_tensors="pt").to(device)

outputs = model(**inputs)
pred = torch.argmax(outputs.logits, dim=1)

print("Positive" if pred == 1 else "Negative")

Positive


# Homework

## DISCUSSION QUESTIONS
- Why does Transformer not need recurrence?
- Why is positional encoding necessary?
- Why does multi-head attention help?
- What happens if we remove attention?
- How is attention different from simple averaging?

## Experiment task

In [ ]:
from datasets import load_dataset

dataset = load_dataset("wikitext", "wikitext-2-raw-v1")

train_texts = dataset["train"]["text"]

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-2-raw-v1/train-00000-of-00001.p(…):   0%|          | 0.00/6.36M [00:00<?, ?B/s]

wikitext-2-raw-v1/validation-00000-of-00(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

In [ ]:
train_texts[110]

' Perhaps the most illuminating points of the above " Summary of Work " and those for following months are that the standard ammunition made was . " buck & ball " , indicating that the .69 caliber smoothbores and shotguns remained the predominant caliber weapon in use , and of this , nearly one sixth or more of all small arms ammunition was still for flintlock weapons , indicating that no less than a sixth of the Confederate troops in this vicinity were still armed with obsolete flintlock weapons . \n'

## STEP 1 — CLEAN THE DATA

Raw text data (e.g., from WikiText) often contains:
- Empty lines
- Extra spaces
- Inconsistent formatting
→ Therefore, we first clean the data.

What we do:
- Remove empty lines
- Strip unnecessary spaces
- Convert all text to lowercase

Question: Why this matters?


## STEP 2 — BUILD A CORPUS

After cleaning, we combine all sentences into one long sequence of text.

Process:
- Join multiple sentences into a single string
- Split into individual words (tokens)

Important note:
- We only use a subset of the dataset (e.g., first 10000 lines)

## STEP 3 — CREATE TRAINING DATA

We convert the text into a next-word prediction task.

Idea:
- Given a sequence of words → predict the next word

Example:
Input:  [w1, w2, w3]
Target: w4

Intuition:
- The model learns context
- It understands how words follow each other


## STEP 4 — BUILD VOCABULARY

We create a mapping between words and numbers.

Two mappings:
- word → index (for input) ("i love nlp" → [3, 10, 25])
- index → word (for decoding output)

## STEP 5 — Model

```python
Input (token ids)
→ Embedding (after embedding→ [[0.2, ...], [0.5, ...], [0.1, ...]])
→ Positional Encoding (Final input = embedding + positional_encoding)
→ Transformer Block:
    → Multi-head Self-Attention
        Q, K, V = Linear(X)
        scores = (Q @ K^T) / sqrt(d_k)
        weights = softmax(scores)
        attention_output = weights @ V
        concat heads → Linear
    → Add & Norm
        x = LayerNorm(x + attention_output)
    → Feed Forward
        ff_output = Linear → ReLU → Linear
    → Add & Norm
        x = LayerNorm(x + ff_output)

→ Pooling / Last Token
→ Linear Layer
→ Softmax
→ Output (prediction)
```

## STEP 6 — TRAINING THE MODEL

Now we train the Mini Transformer.

Training process:

For each (input, target) pair:

- Convert input sequence → indices
- Feed into model
- Model predicts next word
- Compare with true target
- Compute loss
- Backpropagate error
- Update model weights

## STEP 7 — REPORT & EXPLAIN THE PIPELINE
REPORT STRUCTURE

Write a short report following the struction:

🔹 1. Problem Description
- What problem are you solving?
- What is the input?
- What is the output?

🔹 2. Data Processing
- Dataset used
- Cleaning steps

🔹 3. Model Architecture (VERY IMPORTANT)

- Explain your Mini Transformer

🔹 4. Results

🔹 5. Error Analysis
- When does the model fail?
- Why does it fail?

🔹 6. Improvements
must try at least ONE:
- Increase number of heads
- Increase embedding size
- Use more data
- Train longer

🔹 7. Conclusion
- What did you learn?
- What worked well?
- What could be improved?